In [1]:
import os
import pandas as pd
import numpy as np
import re

caminho = "Dados climáticos INMET"
os.makedirs("processados", exist_ok=True)
dados = []

def padronizar_municipio(nome):
    if re.search(r'porto alegre', nome, flags=re.IGNORECASE):
        return 'PORTO ALEGRE'
    return nome.strip().upper()

for raiz, pastas, arquivos in os.walk(caminho):
    
    for arquivo in arquivos:
            caminho_completo = os.path.join(raiz, arquivo)
            
            cabecalho = pd.read_csv(caminho_completo, sep=";", encoding="latin1", on_bad_lines="skip")
            df = pd.read_csv(caminho_completo, sep=";", encoding="latin1", 
                             decimal=",", skiprows=8,  on_bad_lines="skip")
       
            df.drop(columns=[
                            'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
                            'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
                            'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
                            'RADIACAO GLOBAL (Kj/m�)',
                            'RADIACAO GLOBAL (Kj/m²)',
                            'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
                            'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)',
                            'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)',
                            'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)',
                            'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
                            'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
                            'UMIDADE RELATIVA DO AR, HORARIA (%)',
                            'VENTO, DIREÇÃO HORARIA (gr) (° (gr))', 
                            'VENTO, RAJADA MAXIMA (m/s)',
                            'VENTO, VELOCIDADE HORARIA (m/s)',
                            'Unnamed: 19'
                            ], inplace=True, errors='ignore' )
            
            dados.append({
                "arquivo": arquivo,
                "cabecalho": cabecalho,
                "dados": df
            })
        


In [2]:
resultados_lista = []   
     
for item in dados:
    df = item["dados"].copy()
    cabecalho = item['cabecalho']
    
    #pega o nome do municipio
    nome_municipio = cabecalho.iloc[1, 1]
    
    #separa a coluna de data e converte para datetime
    col_data = df.columns[0]
    df[col_data] = pd.to_datetime(df[col_data], errors='coerce')
    
    #desmembra a data em ano e mês, e adiciona o nome do município
    df['ano'] = df[col_data].dt.year
    df['mes'] = df[col_data].dt.month
    df['dia'] = df[col_data].dt.day
    df['municipio'] = nome_municipio
    
    #dropa as colunas desnecessarias com nomes variantes
    df_nova = df.drop(df.columns[[0, 1, 3]], axis=1, errors='ignore') 
    #limpa os nomes das colunas
    df_nova.columns = df_nova.columns.str.strip()
    nome_municipio = nome_municipio.strip().upper()

    #renomeia as colunas para nomes padronizados
    novos_nomes = {df_nova.columns[0]: 'precipitacao', df_nova.columns[1]: 'temp_orvalho'}
    df_nova = df_nova.rename(columns=novos_nomes)
    
    #converte as colunas de precipitação e temperatura de orvalho para numéricas, tratando os erros e arredondando para 2 casas decimais    
    df_nova['precipitacao'] = pd.to_numeric(df_nova['precipitacao'], errors='coerce').round(2)
    df_nova['temp_orvalho'] = pd.to_numeric(df_nova['temp_orvalho'], ).round(2)
    
    #substitui os valores -9999 por NaN e trata os valores negativos de precipitação
    df_nova = df_nova.replace(-9999, 0)
  

    #agrupa por município, ano e mês, calculando a soma,
    # mediana e máximo da precipitação, e a mediana, mínimo e máximo da temperatura de orvalho
    
    df_agrupada = df_nova.groupby(['municipio', 'ano', 'mes', 'dia']).agg({
        'precipitacao': ['sum','median', 'max'],
        'temp_orvalho': ['median', 'min', 'max']
    }).reset_index()
    
    #renomeia as colunas para um formato mais simples
    df_agrupada.columns = [
    '_'.join([c for c in col if c]).strip() if isinstance(col, tuple) else col
    for col in df_agrupada.columns
]

    #ordena os dados por município, ano, mês e dia
    df_agrupada = df_agrupada.sort_values(['municipio', 'ano', 'mes', 'dia'])
    
    #trata os valores faltantes usando interpolação e preenchimento para frente e para trás
    '''"Dados Respiratorios"
    cols = [
            'precipitacao_sum', 'precipitacao_median', 'precipitacao_max',
            'temp_orvalho_median', 'temp_orvalho_min', 'temp_orvalho_max'
        ]

    df_agrupada[cols] = df_agrupada[cols].interpolate()
    df_agrupada[cols] = df_agrupada[cols].ffill().bfill()
    '''
    resultados_lista.append(df_agrupada)
    


In [3]:
#concatena os resultados em um unico dataframe
df_final = pd.concat(resultados_lista, ignore_index=True)
#cria uma coluna de data usando as colunas de datas
df_final['data'] = pd.to_datetime(
    dict(year=df_final['ano'], month=df_final['mes'], day=df_final['dia'])
)
#dropa as colunas de ano, mês e dia, pois agora temos a coluna de data completa
df_final= df_final.drop(columns=['ano', 'mes', 'dia'], errors='ignore')
#ordena os dados por município, ano, mês e dia
df_final = df_final.sort_values('data')
cols_numericas = [
    'precipitacao_sum', 'precipitacao_median', 'precipitacao_max',
    'temp_orvalho_median', 'temp_orvalho_min', 'temp_orvalho_max'
]

df_final[cols_numericas] = df_final[cols_numericas].round(2)
#salva o dataframe final em um arquivo csv
df_final['municipio'] = df_final['municipio'].apply(padronizar_municipio)
df_final.to_csv("processados/dados_climaticos_tratados_novos.csv", index=False, sep=";", encoding="latin1", decimal=",")

display(df_final.head(10))

,municipio,precipitacao_sum,precipitacao_median,precipitacao_max,temp_orvalho_median,temp_orvalho_min,temp_orvalho_max,data
0,PORTO ALEGRE,3.0,0.0,2.6,26.40,22.2,27.4,2015-01-01
365,RIO GRANDE,69.8,0.0,36.2,25.00,20.0,30.7,2015-01-01
7300,FREDERICO WESTPHALEN,0.0,0.0,0.0,19.20,17.9,24.9,2015-01-01
8395,VACARIA,8.2,0.1,1.8,18.50,17.8,21.0,2015-01-01
6935,CRUZ ALTA,59.2,1.2,13.0,19.80,17.9,23.0,2015-01-01
730,SANTA MARIA,24.4,0.0,9.8,23.75,18.9,27.2,2015-01-01
6570,LAGOA VERMELHA,25.4,0.8,5.8,18.70,17.6,21.8,2015-01-01
6205,BENTO GONCALVES,1.6,0.0,0.8,21.55,19.3,22.8,2015-01-01
1095,SANTO AUGUSTO,95.8,3.8,15.2,19.55,17.9,24.3,2015-01-01
5840,PASSO FUNDO,37.2,0.5,5.8,19.40,17.5,22.2,2015-01-01


In [7]:
df_final_anual = df_final.drop(columns={'precipitacao_median', 'precipitacao_max', 'temp_orvalho_min', 'temp_orvalho_max'}, errors='ignore')
df_final_anual['ano'] = df_final_anual['data'].dt.year

df_final_mes = df_final_anual.copy()
df_final_mes['mes'] = df_final_anual['data'].dt.month

df_final_anual = df_final_anual.groupby(['municipio', 'ano']).agg({'precipitacao_sum': 'sum', 'temp_orvalho_median': 'median'}).reset_index().round(3)
display(df_final_anual)

df_final_mes = df_final_mes.groupby(['municipio', 'ano', 'mes']).agg({'precipitacao_sum': 'sum', 'temp_orvalho_median': 'median'}).reset_index().round(3)
display(df_final_mes)


df_final_anual.to_csv("processados/dados_climaticos_tratados_anual.csv", index=False, sep=";", encoding="latin1", decimal=".")
df_final_mes.to_csv("processados/dados_climaticos_tratados_mensal.csv", index=False, sep=";", encoding="latin1", decimal=".")


,municipio,ano,precipitacao_sum,temp_orvalho_median
0,BAGE,2015,1980.4,17.600
1,BAGE,2016,1175.8,17.375
2,BAGE,2017,1535.6,17.600
3,BAGE,2018,1270.2,18.250
4,BAGE,2019,1715.8,18.350
...,...,...,...,...
289,VACARIA,2021,1471.6,12.100
290,VACARIA,2022,1898.4,11.750
291,VACARIA,2023,1769.6,12.700
292,VACARIA,2024,2175.0,13.700


,municipio,ano,mes,precipitacao_sum,temp_orvalho_median
0,BAGE,2015,1,207.4,22.300
1,BAGE,2015,2,87.8,21.675
2,BAGE,2015,3,32.4,21.300
3,BAGE,2015,4,21.2,18.175
4,BAGE,2015,5,126.2,14.750
...,...,...,...,...,...
3514,VACARIA,2025,8,113.8,8.950
3515,VACARIA,2025,9,223.6,10.175
3516,VACARIA,2025,10,138.4,11.750
3517,VACARIA,2025,11,124.8,12.875
